# 05 — Dataset Assembly
Converts validated samples to ChatML format, deduplicates, balances task types,
and writes train / valid / test splits.
Also writes an `mlx/` subfolder with the exact format expected by `mlx_lm.lora`.

**Input:**  `../data/04_validated/valid_samples.jsonl`
**Output:** `../data/05_dataset/train.jsonl`  (full format with metadata)
            `../data/05_dataset/valid.jsonl`   (renamed from val — mlx-lm convention)
            `../data/05_dataset/test.jsonl`
            `../data/05_dataset/mlx/train.jsonl`   ({"messages":[...]} only)
            `../data/05_dataset/mlx/valid.jsonl`
            `../data/05_dataset/stats.json`

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

DATA_IN  = SCRIPT_DIR / '../data/04_validated'
DATA_OUT = SCRIPT_DIR / '../data/05_dataset'
MLX_DIR  = DATA_OUT / 'mlx'
DATA_OUT.mkdir(parents=True, exist_ok=True)
MLX_DIR.mkdir(parents=True, exist_ok=True)

samples = []
with open(DATA_IN / 'valid_samples.jsonl') as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f'Loaded {len(samples)} valid samples')
print('Distribution:', dict(Counter(s['task_type'] for s in samples)))

## System prompt

In [ ]:
ARO_SYSTEM_PROMPT = """\
You are an expert ARO language programmer. \
ARO (Action Result Object) is a domain-specific language for expressing business logic \
as natural-language Action-Result-Object statements.

Core syntax:
  (FeatureSetName: BusinessActivity) {
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }

Key rules: \
string concat uses ++, \
variables are hyphenated lowercase like <user-id>, \
HTTP apps require openapi.yaml, \
Application-Start is the required entry point, \
Return statuses: <OK: status> <Created: status> <NotFound: status>.\
"""

## Convert to ChatML

In [ ]:
def build_messages(sample):
    task  = sample.get('task_type', '')
    instr = sample.get('instruction', '').strip()
    inp   = sample.get('input', '').strip()
    out   = sample.get('output', '').strip()

    if not out:
        return None

    if task == 'fim':
        prefix = sample.get('prefix', '')
        suffix = sample.get('suffix', '')
        middle = sample.get('middle', '')
        if not middle.strip():
            return None
        user = f'Complete the missing ARO statement(s). Only output the missing line(s).\n\n```aro\n{prefix}\n<FILL_HERE>\n{suffix}\n```'
        return user, middle

    if inp:
        user = f'{instr}\n\n```aro\n{inp}\n```'
    else:
        user = instr

    return (user, out) if user.strip() else None


def to_chatml(sample):
    result = build_messages(sample)
    if result is None:
        return None
    user_content, assistant_content = result
    if len(user_content) < 10 or len(assistant_content) < 10:
        return None
    return {
        'messages': [
            {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',      'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ],
        'task_type': sample.get('task_type', ''),
        'source':    sample.get('source', sample.get('domain', sample.get('scenario', ''))),
    }


chatml = [to_chatml(s) for s in samples]
chatml = [c for c in chatml if c is not None]
print(f'Converted: {len(chatml)}  (dropped {len(samples) - len(chatml)} empty)')

## Deduplicate

In [ ]:
seen, deduped = set(), []
for s in chatml:
    key = s['messages'][1]['content'][:300]
    if key not in seen:
        seen.add(key)
        deduped.append(s)
print(f'After dedup: {len(deduped)}')

## Cap per-task count (max 40% per type)

In [ ]:
MAX_PER_TYPE = max(500, len(deduped) // 3)
capped, type_counts = [], Counter()
for s in deduped:
    t = s['task_type']
    if type_counts[t] < MAX_PER_TYPE:
        capped.append(s)
        type_counts[t] += 1

print(f'After cap: {len(capped)}')
for t, n in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {t:25s}: {n}')

## Train / valid / test split  (90 / 5 / 5)

In [ ]:
random.seed(42)
random.shuffle(capped)
n     = len(capped)
t_end = int(n * 0.90)
v_end = int(n * 0.95)
train = capped[:t_end]
valid = capped[t_end:v_end]
test  = capped[v_end:]
print(f'train={len(train)}  valid={len(valid)}  test={len(test)}')

## Save full format + mlx format

In [ ]:
def save_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    print(f'  {path.name}: {len(data)} samples')


# Full format (with task_type / source metadata)
save_jsonl(train, DATA_OUT / 'train.jsonl')
save_jsonl(valid, DATA_OUT / 'valid.jsonl')   # NOTE: valid (not val) — mlx-lm convention
save_jsonl(test,  DATA_OUT / 'test.jsonl')

# mlx-lm format: only the "messages" key (mlx_lm.lora --use-chat-template reads this)
save_jsonl([{'messages': s['messages']} for s in train], MLX_DIR / 'train.jsonl')
save_jsonl([{'messages': s['messages']} for s in valid], MLX_DIR / 'valid.jsonl')

# Stats
stats = {
    'total': len(capped), 'train': len(train), 'valid': len(valid), 'test': len(test),
    'task_counts': dict(type_counts),
    'avg_user_len':      int(sum(len(s['messages'][1]['content']) for s in capped) / max(1, len(capped))),
    'avg_assistant_len': int(sum(len(s['messages'][2]['content']) for s in capped) / max(1, len(capped))),
}
with open(DATA_OUT / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\nstats.json: avg user={stats["avg_user_len"]} chars, avg assistant={stats["avg_assistant_len"]} chars')

## Sanity check

In [ ]:
for s in random.sample(train, 3):
    print('─' * 60)
    print(f'Task: {s["task_type"]}')
    print(f'User:      {s["messages"][1]["content"][:120]}...')
    print(f'Assistant: {s["messages"][2]["content"][:120]}...')
print('─' * 60)